In [59]:
"""
Roboflow Metrics Evaluation Notebook

This notebook reads predictions from Roboflow (JSON format) and original labels (YOLO format),
then compares them to calculate mAP, precision, and recall metrics.

USAGE:
1. Update the configuration in Cell 1 with your paths and class names
2. Run all cells in order
3. Results will be saved to CSV and markdown report in the OUTPUT_DIR

REQUIREMENTS:
- Roboflow prediction JSON files in PREDICTIONS_DIR
- YOLO format label files (.txt) in LABELS_DIR
- Class names must match the order in your YOLO labels (class_id 0, 1, 2, ...)
"""

import os
import json
import cv2
import numpy as np
from tqdm import tqdm
import supervision as sv
import pandas as pd
from datetime import datetime

print("✅ Imports successful")


✅ Imports successful


In [60]:
# --- CONFIGURATION ---
# Update these paths according to your setup

PREDICTIONS_DIR = "/home/emma/facultad/pps/validacion/ppe-detect-yowms/ppe-detection-yolov8-dataset-train/predictions"  # Directory containing Roboflow prediction JSON files
LABELS_DIR = "/home/emma/facultad/pps/datasets/ppe-detection-yolov8-dataset/train/labels"           # Directory containing YOLO format label files
IMAGES_DIR = "/home/emma/facultad/pps/datasets/ppe-detection-yolov8-dataset/train/images"                     # Optional: Directory with images (for getting image dimensions if not in JSON)

# Class names must match the order used in your YOLO labels (class_id 0, 1, 2, ...)
CLASS_NAMES = ['no-eyewear', 'no-gloves', 'no-mask', 'person'] # Update with your actual class names

# Metrics calculation parameters
IOU_THRESHOLD = 0.5      # IoU threshold for matching detections
CONF_THRESHOLD = 0.25    # Confidence threshold for predictions

# Output directory for results
OUTPUT_DIR = "/home/emma/facultad/pps/validacion/ppe-detect-yowms/ppe-detection-yolov8-dataset-train"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📂 Predictions directory: {PREDICTIONS_DIR}")
print(f"📂 Labels directory: {LABELS_DIR}")
print(f"📂 Output directory: {OUTPUT_DIR}")
print(f"📋 Class names: {CLASS_NAMES}")


📂 Predictions directory: /home/emma/facultad/pps/validacion/ppe-detect-yowms/ppe-detection-yolov8-dataset-train/predictions
📂 Labels directory: /home/emma/facultad/pps/datasets/ppe-detection-yolov8-dataset/train/labels
📂 Output directory: /home/emma/facultad/pps/validacion/ppe-detect-yowms/ppe-detection-yolov8-dataset-train
📋 Class names: ['no-eyewear', 'no-gloves', 'no-mask', 'person']


In [61]:
def read_yolo_labels(label_path, img_shape):
    """
    Read YOLO format labels (x_center, y_center, width, height) normalized -> absolute xyxy.
    
    Args:
        label_path: Path to YOLO format label file (.txt)
        img_shape: Image shape tuple (height, width, channels)
    
    Returns:
        boxes: numpy array of shape (N, 4) with xyxy coordinates
        classes: numpy array of shape (N,) with class IDs
    """
    H, W = img_shape[:2]
    boxes = []
    classes = []
    
    if not os.path.exists(label_path):
        return np.array(boxes), np.array(classes)
    
    with open(label_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = list(map(float, line.strip().split()))
            cls, x, y, w, h = parts[:5]  # ignore polygons if exist
            
            # Convert normalized YOLO format to absolute xyxy
            x1 = (x - w / 2) * W
            y1 = (y - h / 2) * H
            x2 = (x + w / 2) * W
            y2 = (y + h / 2) * H
            boxes.append([x1, y1, x2, y2])
            classes.append(int(cls))
    
    return np.array(boxes), np.array(classes)

print("✅ read_yolo_labels function defined")


✅ read_yolo_labels function defined


In [62]:
def roboflow_to_xyxy(preds):
    """
    Convert Roboflow predictions (x, y, width, height) to xyxy absolute boxes.
    
    Args:
        preds: List of prediction dictionaries from Roboflow JSON
        
    Returns:
        boxes: numpy array of shape (N, 4) with xyxy coordinates
        classes: numpy array of shape (N,) with class IDs
        confs: numpy array of shape (N,) with confidence scores
    """
    boxes, classes, confs = [], [], []
    
    for p in preds:
        # Roboflow format: x, y are center coordinates, width and height are absolute
        x1 = p["x"] - p["width"] / 2
        y1 = p["y"] - p["height"] / 2
        x2 = p["x"] + p["width"] / 2
        y2 = p["y"] + p["height"] / 2
        boxes.append([x1, y1, x2, y2])
        classes.append(p["class_id"])
        confs.append(p["confidence"])
    
    return np.array(boxes), np.array(classes), np.array(confs)

print("✅ roboflow_to_xyxy function defined")


✅ roboflow_to_xyxy function defined


In [63]:
from supervision.metrics.mean_average_precision import MeanAveragePrecision

def calculate_map(detections_pred, detections_gt):
    """
    Calculate mAP (mean average precision) for object detection.
    
    Args:
        detections_pred: sv.Detections object with predictions
        detections_gt: sv.Detections object with ground truth
    
    Returns:
        map_result: MeanAveragePrecisionResult object with mAP metrics
    """
    # Create the metric
    map_metric = MeanAveragePrecision()

    # Add your predictions and ground-truth Detections
    map_metric.update(predictions=detections_pred, targets=detections_gt)

    # Compute results
    return map_metric.compute()

print("✅ calculate_map function defined")


✅ calculate_map function defined


In [64]:
from supervision.metrics.detection import ConfusionMatrix

def calculate_precision_recall(predictions_pred, predictions_gt, class_names, iou_threshold=0.5, conf_threshold=0.25):
    """
    Compute Precision and Recall using Supervision ConfusionMatrix.
    
    Args:
        predictions_pred: sv.Detections object with predictions
        predictions_gt: sv.Detections object with ground truth
        class_names: List of class names
        iou_threshold: IoU threshold for matching detections
        conf_threshold: Confidence threshold for predictions
    
    Returns:
        mean_precision: Mean precision across all classes
        mean_recall: Mean recall across all classes
        mean_f1: Mean F1 score across all classes
    """
    cm = ConfusionMatrix.from_detections(
        predictions=[predictions_pred],
        targets=[predictions_gt],
        classes=class_names,
        conf_threshold=conf_threshold,
        iou_threshold=iou_threshold,
    )
    M = cm.matrix
    # matrix shape = (n_classes+1, n_classes+1)
    # last row/col = background/missed detections

    num_classes = len(class_names)
    tp = np.diag(M[:num_classes, :num_classes])           # true positives per class
    fp = M[:num_classes, num_classes]                     # false positives per class
    fn = M[num_classes, :num_classes]                     # false negatives per class

    precision = tp / (tp + fp + 1e-16)
    recall = tp / (tp + fn + 1e-16)
    f1 = 2 * precision * recall / (precision + recall + 1e-16)

    # Replace NaN (if no samples) with 0
    precision = np.nan_to_num(precision)
    recall = np.nan_to_num(recall)
    f1 = np.nan_to_num(f1)

    mean_precision = precision.mean()
    mean_recall = recall.mean()
    mean_f1 = f1.mean()

    return mean_precision, mean_recall, mean_f1

print("✅ calculate_precision_recall function defined")


✅ calculate_precision_recall function defined


In [65]:
def load_roboflow_prediction(pred_path):
    """
    Load Roboflow prediction JSON file.
    
    Args:
        pred_path: Path to Roboflow prediction JSON file
    
    Returns:
        pred_data: Dictionary with prediction data
        img_width: Image width
        img_height: Image height
    """
    with open(pred_path, "r") as f:
        pred_data = json.load(f)
    
    # Extract image dimensions from Roboflow JSON
    # Roboflow JSON structure: {"image": {"width": ..., "height": ...}, "predictions": [...]}
    if "image" in pred_data:
        img_width = int(pred_data["image"]["width"])
        img_height = int(pred_data["image"]["height"])
    elif len(pred_data.get("predictions", [])) > 0:
        # Fallback: try to get from first prediction if available
        first_pred = pred_data["predictions"][0]
        if "image" in first_pred:
            img_width = int(first_pred["image"]["width"])
            img_height = int(first_pred["image"]["height"])
        else:
            raise ValueError("Image dimensions not found in prediction JSON")
    else:
        raise ValueError("Could not determine image dimensions from prediction JSON")
    
    return pred_data, img_width, img_height

print("✅ load_roboflow_prediction function defined")


✅ load_roboflow_prediction function defined


In [66]:
def get_image_shape_from_file(image_path):
    """
    Get image shape from image file.
    
    Args:
        image_path: Path to image file
    
    Returns:
        img_shape: Tuple (height, width, channels)
    """
    image = cv2.imread(image_path)
    if image is None:
        return None
    return image.shape

print("✅ get_image_shape_from_file function defined")


✅ get_image_shape_from_file function defined


In [67]:
def evaluate_roboflow_predictions(predictions_dir, labels_dir, class_names, images_dir=None, 
                                  iou_threshold=0.5, conf_threshold=0.25):
    """
    Evaluate Roboflow predictions against YOLO-format ground-truth labels.
    
    Args:
        predictions_dir: Directory containing Roboflow prediction JSON files
        labels_dir: Directory containing YOLO format label files
        class_names: List of class names
        images_dir: Optional directory with images (for getting dimensions if not in JSON)
        iou_threshold: IoU threshold for matching detections
        conf_threshold: Confidence threshold for predictions
    
    Returns:
        results: Dictionary with overall and per-image metrics
    """
    
    # Find all prediction JSON files
    pred_files = [
        f for f in os.listdir(predictions_dir)
        if f.endswith(".json")
    ]
    
    print(f"📂 Found {len(pred_files)} prediction files in: {predictions_dir}")
    if not pred_files:
        print("❌ No predictions found.")
        return None
    
    all_pred_detections = []
    all_gt_detections = []
    image_metrics = []
    
    processed = 0
    skipped = 0
    
    for pred_file in tqdm(pred_files, desc="Evaluating images"):
        base_name = os.path.splitext(pred_file)[0]
        pred_path = os.path.join(predictions_dir, pred_file)
        
        # Try to find corresponding label file
        # First try exact match
        label_path = os.path.join(labels_dir, f"{base_name}.txt")
        
        # If not found, try removing Roboflow suffixes (e.g., .rf.xxx)
        if not os.path.exists(label_path):
            # Remove Roboflow hash suffix if present
            base_without_rf = base_name.split('.rf.')[0] if '.rf.' in base_name else base_name
            label_path = os.path.join(labels_dir, f"{base_without_rf}.txt")
        
        if not os.path.exists(label_path):
            print(f"⚠️ Missing label for {base_name}")
            skipped += 1
            continue
        
        try:
            # Load prediction
            pred_data, img_width, img_height = load_roboflow_prediction(pred_path)
            img_shape = (img_height, img_width, 3)
            
            preds = pred_data.get("predictions", [])
            if not preds:
                # Empty predictions are valid - just skip for metrics but note it
                pred_boxes = np.array([]).reshape(0, 4)
                pred_classes = np.array([])
                pred_conf = np.array([])
            else:
                pred_boxes, pred_classes, pred_conf = roboflow_to_xyxy(preds)
            
            # Load ground truth labels
            gt_boxes, gt_classes = read_yolo_labels(label_path, img_shape)
            
            # Build supervision Detections
            if len(gt_boxes) > 0:
                gt_det = sv.Detections(xyxy=gt_boxes, class_id=gt_classes)
            else:
                gt_det = sv.Detections.empty()
            
            if len(pred_boxes) > 0:
                pred_det = sv.Detections(xyxy=pred_boxes, confidence=pred_conf, class_id=pred_classes)
            else:
                pred_det = sv.Detections.empty()
            
            # Calculate per-image metrics
            if len(pred_boxes) > 0 and len(gt_boxes) > 0:
                precision, recall, f1 = calculate_precision_recall(
                    pred_det, gt_det, class_names, iou_threshold, conf_threshold
                )
            else:
                # If no predictions or no ground truth, set metrics appropriately
                if len(pred_boxes) == 0 and len(gt_boxes) == 0:
                    precision, recall, f1 = 1.0, 1.0, 1.0  # Both empty = perfect match
                elif len(pred_boxes) == 0:
                    precision, recall, f1 = 1.0, 0.0, 0.0  # No predictions but GT exists = perfect precision, zero recall
                else:  # len(gt_boxes) == 0
                    precision, recall, f1 = 0.0, 1.0, 0.0  # Predictions but no GT = zero precision, perfect recall
            
            image_metrics.append({
                "image": base_name,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "num_pred": len(pred_boxes),
                "num_gt": len(gt_boxes),
            })
            
            # Store detections for global metrics
            all_pred_detections.append(pred_det)
            all_gt_detections.append(gt_det)
            
            processed += 1
            
        except Exception as e:
            print(f"❌ Error processing {base_name}: {str(e)}")
            skipped += 1
            continue
    
    print(f"\n📊 Processed: {processed}, Skipped: {skipped}")
    
    if not all_pred_detections or not all_gt_detections:
        print("❌ No valid detections for evaluation.")
        return None
    
    # Merge detections for global metrics
    def merge_detections(dets_list):
        """Concatenate multiple sv.Detections into one."""
        if not dets_list:
            return sv.Detections.empty()
        dets_with_data = [d for d in dets_list if len(d.xyxy) > 0]
        if not dets_with_data:
            return sv.Detections.empty()
        
        xyxy = np.concatenate([d.xyxy for d in dets_with_data], axis=0)
        class_id = np.concatenate([d.class_id for d in dets_with_data if d.class_id is not None], axis=0)
        confidence = np.concatenate(
            [d.confidence for d in dets_with_data if d.confidence is not None], axis=0
        ) if any(d.confidence is not None for d in dets_with_data) else None
        return sv.Detections(xyxy=xyxy, class_id=class_id, confidence=confidence)
    
    merged_pred = merge_detections(all_pred_detections)
    merged_gt = merge_detections(all_gt_detections)
    
    # Calculate global metrics
    map_result = calculate_map(merged_pred, merged_gt)
    overall_precision, overall_recall, overall_f1 = calculate_precision_recall(
        merged_pred, merged_gt, class_names, iou_threshold, conf_threshold
    )
    
    # Compile results
    results = {
        "overall_metrics": {
            "mAP@50": getattr(map_result, "map50", np.nan),
            "mAP@75": getattr(map_result, "map75", np.nan),
            "mAP@50-95": getattr(map_result, "map", np.nan),
            "precision": overall_precision,
            "recall": overall_recall,
            "f1": overall_f1,
            "total_images": processed,
            "skipped_images": skipped,
        },
        "per_image_metrics": image_metrics,
        "class_names": class_names,
    }
    
    print("\n✅ Evaluation complete.")
    print(f"📈 Overall Precision: {overall_precision:.4f}")
    print(f"📈 Overall Recall: {overall_recall:.4f}")
    print(f"📈 Overall F1: {overall_f1:.4f}")
    print(f"📈 mAP@50: {map_result.map50:.4f}")
    print(f"📈 mAP@75: {map_result.map75:.4f}")
    
    return results

print("✅ evaluate_roboflow_predictions function defined")


✅ evaluate_roboflow_predictions function defined


In [68]:
# Run the evaluation
# Make sure to update the configuration in cell 1 before running this!

results = evaluate_roboflow_predictions(
    predictions_dir=PREDICTIONS_DIR,
    labels_dir=LABELS_DIR,
    class_names=CLASS_NAMES,
    images_dir=IMAGES_DIR if os.path.exists(IMAGES_DIR) else None,
    iou_threshold=IOU_THRESHOLD,
    conf_threshold=CONF_THRESHOLD
)


📂 Found 521 prediction files in: /home/emma/facultad/pps/validacion/ppe-detect-yowms/ppe-detection-yolov8-dataset-train/predictions


Evaluating images: 100%|██████████| 521/521 [00:00<00:00, 1718.28it/s]



📊 Processed: 521, Skipped: 0

✅ Evaluation complete.
📈 Overall Precision: 0.3428
📈 Overall Recall: 0.3540
📈 Overall F1: 0.3481
📈 mAP@50: 0.2073
📈 mAP@75: 0.0889


In [69]:
# Save results to CSV
if results:
    df = pd.DataFrame(results["per_image_metrics"])
    csv_path = os.path.join(OUTPUT_DIR, "evaluation_report.csv")
    df.to_csv(csv_path, index=False)
    print(f"💾 Saved detailed metrics to {csv_path}")
    
    # Display summary
    print("\n📊 Summary Statistics:")
    print(df.describe())
else:
    print("❌ No results to save")


💾 Saved detailed metrics to /home/emma/facultad/pps/validacion/ppe-detect-yowms/ppe-detection-yolov8-dataset-train/evaluation_report.csv

📊 Summary Statistics:
        precision      recall          f1    num_pred      num_gt
count  521.000000  521.000000  521.000000  521.000000  521.000000
mean     0.333734    0.300848    0.290184    3.023033    2.483685
std      0.228128    0.197514    0.186631    2.726552    1.783531
min      0.000000    0.000000    0.000000    0.000000    1.000000
25%      0.250000    0.250000    0.250000    1.000000    1.000000
50%      0.250000    0.250000    0.250000    2.000000    2.000000
75%      0.500000    0.375000    0.416667    4.000000    4.000000
max      1.000000    0.750000    0.750000   15.000000   11.000000


In [70]:
# Generate markdown report
if results:
    def generate_markdown_report(results, dataset_path, output_dir):
        """Generate a markdown report with evaluation results."""
        if not results:
            print("❌ No results to generate report")
            return None
        
        df = pd.DataFrame(results["per_image_metrics"])
        
        # Calculate summary statistics
        metrics = {
            "Precision": {
                "mean": df["precision"].mean(),
                "std": df["precision"].std(),
                "min": df["precision"].min(),
                "max": df["precision"].max()
            },
            "Recall": {
                "mean": df["recall"].mean(),
                "std": df["recall"].std(),
                "min": df["recall"].min(),
                "max": df["recall"].max()
            },
            "F1": {
                "mean": df["f1"].mean(),
                "std": df["f1"].std(),
                "min": df["f1"].min(),
                "max": df["f1"].max()
            }
        }
        
        # Generate timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # Generate markdown content
        markdown = f"""# Roboflow Predictions Evaluation Report

**Generated on:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

## Dataset Information
- **Predictions Directory:** {PREDICTIONS_DIR}
- **Labels Directory:** {LABELS_DIR}
- **Total Images Processed:** {results['overall_metrics']['total_images']}
- **Skipped Images:** {results['overall_metrics']['skipped_images']}

## Overall Metrics

| Metric | Value |
|--------|-------|
| mAP@50 | {results['overall_metrics']['mAP@50']:.4f} |
| mAP@75 | {results['overall_metrics']['mAP@75']:.4f} |
| mAP@50-95 | {results['overall_metrics']['mAP@50-95']:.4f} |
| Precision | {results['overall_metrics']['precision']:.4f} |
| Recall | {results['overall_metrics']['recall']:.4f} |
| F1 Score | {results['overall_metrics']['f1']:.4f} |

## Per-Image Statistics

| Metric | Mean | Std Dev | Min | Max |
|--------|------|---------|-----|-----|
| Precision | {metrics['Precision']['mean']:.3f} | {metrics['Precision']['std']:.3f} | {metrics['Precision']['min']:.3f} | {metrics['Precision']['max']:.3f} |
| Recall | {metrics['Recall']['mean']:.3f} | {metrics['Recall']['std']:.3f} | {metrics['Recall']['min']:.3f} | {metrics['Recall']['max']:.3f} |
| F1 | {metrics['F1']['mean']:.3f} | {metrics['F1']['std']:.3f} | {metrics['F1']['min']:.3f} | {metrics['F1']['max']:.3f} |

## Class Names
{', '.join(results['class_names'])}

## Evaluation Parameters
- **IoU Threshold:** {IOU_THRESHOLD}
- **Confidence Threshold:** {CONF_THRESHOLD}
"""
        
        # Save markdown report
        report_filename = f"roboflow_evaluation_report_{timestamp}.md"
        report_path = os.path.join(output_dir, report_filename)
        
        with open(report_path, "w") as f:
            f.write(markdown)
        
        print(f"📄 Generated markdown report: {report_filename}")
        print(f"📊 Report saved to: {report_path}")
        
        return report_path
    
    report_path = generate_markdown_report(results, PREDICTIONS_DIR, OUTPUT_DIR)
    if report_path:
        print("✅ Markdown report generated successfully!")


📄 Generated markdown report: roboflow_evaluation_report_20251106_085740.md
📊 Report saved to: /home/emma/facultad/pps/validacion/ppe-detect-yowms/ppe-detection-yolov8-dataset-train/roboflow_evaluation_report_20251106_085740.md
✅ Markdown report generated successfully!
